# Medallion Architecture on Databricks - Retail Sales

- **Bronze:** raw CSV data (loaded from a Unity Catalog volume) stored as Delta tables
- **Silver:** cleansed, standardized, validated, and enriched Delta tables
- **Gold:** business-ready sales metrics and summaries

**Dataset:** customers, products, orders, and order items.

> Run the notebook from top to bottom on Databricks. The setup cell creates a catalog named `sales_demo` and three schemas: `bronze`, `silver`, and `gold`.

## 0. Prerequisites

You need a Databricks free account.

The raw source data ships as external CSV files in the `data/` folder next to this notebook:

- `data/customers.csv`
- `data/products.csv`
- `data/orders.csv`
- `data/order_items.csv`

Before running section 1, upload these four files to the Unity Catalog volume created by the setup cell (`/Volumes/sales_demo/bronze/raw_files`) using **Catalog Explorer → Upload to this volume**.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import date

CATALOG = "sales_demo"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"
VOLUME_PATH = f"/Volumes/{CATALOG}/bronze/raw_files"

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS sales_demo;
CREATE SCHEMA IF NOT EXISTS sales_demo.bronze;
CREATE SCHEMA IF NOT EXISTS sales_demo.silver;
CREATE SCHEMA IF NOT EXISTS sales_demo.gold;
CREATE VOLUME IF NOT EXISTS sales_demo.bronze.raw_files;

## 1. Load Source Data from the Volume

The raw records intentionally include common quality problems:

- Duplicate customers and order items
- Blank email and category values
- Inconsistent country and category labels
- Numeric values stored as text
- A malformed price
- An unknown customer and product
- A future order date
- Negative quantity

Each dataset is read directly from the CSV files uploaded to the `raw_files` volume.

In [0]:
# Load raw CSV files from the Unity Catalog volume into a dictionary of DataFrames
# Each dataset (customers, products, orders, order_items) is read from the raw_files volume
# header=True: First row contains column names
# inferSchema=False: Keep all columns as strings to preserve raw data exactly as-is
bronze_inputs = {
    name: spark.read.option("header", True).option("inferSchema", False)
        .csv(f"{VOLUME_PATH}/{name}.csv")
    for name in ("customers", "products", "orders", "order_items")
}

# Display each raw DataFrame to inspect the source data quality
# This helps identify duplicates, nulls, inconsistent formats, and other issues
# that will be cleaned in the Silver layer
for name, df in bronze_inputs.items():
    display(df)

customer_id,customer_name,email,country,updated_at
1,ali hassan,ali@example.com,Qatar,2026-01-01 10:00:00
2,SARA AHMED,null,QA,2026-01-02 11:00:00
2,Sara Ahmed,sara@example.com,QAT,2026-01-03 12:00:00
3,Omar Saleh,omar@example.com,qatar,2026-01-04 09:00:00


product_id,product_name,category,unit_price
101,Laptop,electronics,1500.00
102,Mouse,null,25.00
103,Keyboard,ELECTRONICS,49.99
104,Desk,Office Furniture,bad_price


order_id,customer_id,order_date,status
1001,1,2026-01-15,completed
1002,2,2026-01-16,Complete
1003,99,2026-01-17,completed
1004,3,2099-01-01,pending


order_id,product_id,quantity
1001,101,1
1001,102,2
1001,102,2
1002,103,3
1002,999,1
1003,103,-1


## 2. Bronze Layer: Preserve Raw Data

Bronze stores the source data with minimal change. We add ingestion metadata for traceability, then save each dataset as a managed Delta table.

In [0]:
# Loop through each source dataset (customers, products, orders, order_items)
for name, df in bronze_inputs.items():
    # Add metadata columns to track data lineage and ingestion time
    bronze_df = (
        df.withColumn("_ingested_at", F.current_timestamp())  # Timestamp when data was loaded
          .withColumn("_source", F.lit(f"{VOLUME_PATH}/{name}.csv"))  # Source file path
    )
    # Write each DataFrame as a managed Delta table in the Bronze schema
    # overwrite mode: replace existing data completely
    # overwriteSchema: allow schema changes between loads
    (bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{BRONZE}.{name}"))

In [0]:
%sql
SHOW TABLES IN sales_demo.bronze;

database,tableName,isTemporary
bronze,customers,false
bronze,order_items,false
bronze,orders,false
bronze,products,false


## 3. Profile Bronze Data

Inspect row counts, schemas, nulls, duplicates, and invalid values before defining Silver rules.

In [0]:
# Display row counts and schemas for all Bronze tables
for table in ("customers", "products", "orders", "order_items"):
    df = spark.table(f"{BRONZE}.{table}")
    print(f"{table}: {df.count()} rows")
    df.printSchema()

# Identify duplicate customer records
print("Duplicate customer IDs:")
(spark.table(f"{BRONZE}.customers")
 .groupBy("customer_id").count()
 .filter("count > 1")
 .show())

# Find products with malformed prices that cannot be converted to decimal
print("Invalid product prices:")
(spark.table(f"{BRONZE}.products")
 .filter(F.col("unit_price").cast("decimal(10,2)").isNull())
 .show())

customers: 4 rows
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- updated_at: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source: string (nullable = true)

products: 4 rows
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source: string (nullable = true)

orders: 4 rows
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source: string (nullable = true)

order_items: 6 rows
root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: string (nullabl

## 4. Silver Layer: Clean and Standardize Customers

Rules:

- Cast the business key and timestamp to proper types.
- Trim and format names.
- Convert blank emails to null.
- Standardize country values to `QAT`.
- Keep the latest customer record for each `customer_id`.

In [0]:
# Load raw customer data from Bronze layer
customers_raw = spark.table(f"{BRONZE}.customers")

# Clean and standardize customer fields
customers_staged = (
    customers_raw
    .select(
        F.col("customer_id").cast("int").alias("customer_id"),  # Cast ID to integer
        F.initcap(F.trim("customer_name")).alias("customer_name"),  # Trim and title case names
        F.when(F.trim("email") == "", None)  # Convert blank emails to null
         .otherwise(F.lower(F.trim("email"))).alias("email"),  # Lowercase valid emails
        F.upper(F.trim("country")).alias("country_raw"),  # Uppercase country for matching
        F.to_timestamp("updated_at").alias("updated_at"),  # Parse timestamp
        "_ingested_at"
    )
    .withColumn(
        "country_code",
        F.when(F.col("country_raw").isin("QATAR", "QA", "QAT"), "QAT")  # Standardize Qatar codes
         .otherwise(F.col("country_raw"))
    )
)

# Define window to identify the most recent record per customer
latest_customer = Window.partitionBy("customer_id").orderBy(
    F.col("updated_at").desc_nulls_last(), F.col("_ingested_at").desc()
)

# Deduplicate: keep only the latest record for each customer
customers_silver = (
    customers_staged
    .withColumn("rn", F.row_number().over(latest_customer))  # Assign row numbers
    .filter("rn = 1 AND customer_id IS NOT NULL")  # Keep first row per customer with valid ID
    .drop("rn", "country_raw")  # Remove helper columns
)

# Write deduplicated customers to Silver layer
(customers_silver.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.customers"))
display(customers_silver)

customer_id,customer_name,email,updated_at,_ingested_at,country_code
1,Ali Hassan,ali@example.com,2026-01-01T10:00:00.000Z,2026-09-08T17:52:26.690Z,QAT
2,Sara Ahmed,sara@example.com,2026-01-03T12:00:00.000Z,2026-09-08T17:52:26.690Z,QAT
3,Omar Saleh,omar@example.com,2026-01-04T09:00:00.000Z,2026-09-08T17:52:26.690Z,QAT


## 5. Silver Layer: Clean Products and Quarantine Invalid Rows

Rules:

- Cast identifiers and prices.
- Standardize category casing.
- Replace blank categories with `Unknown`.
- Quarantine rows with invalid IDs or prices instead of silently dropping them.

In [0]:
# Load raw product data from Bronze layer
products_raw = spark.table(f"{BRONZE}.products")

# Clean and standardize product fields
products_staged = (
    products_raw
    .select(
        F.col("product_id").cast("int").alias("product_id"),  # Cast ID to integer
        F.initcap(F.trim("product_name")).alias("product_name"),  # Trim and title case names
        F.when(F.trim("category") == "", "Unknown")  # Replace blank categories with "Unknown"
         .otherwise(F.initcap(F.trim("category"))).alias("category"),
        F.expr("try_cast(unit_price as decimal(10,2))").alias("unit_price"),  # Cast price to decimal, malformed values become NULL
        "_ingested_at"  # Preserve metadata
    )
)

# Separate valid products (non-null ID and price, price >= 0)
valid_products = products_staged.filter(
    "product_id IS NOT NULL AND unit_price IS NOT NULL AND unit_price >= 0"
)
# Quarantine invalid products for review
invalid_products = (
    products_staged
    .filter("product_id IS NULL OR unit_price IS NULL OR unit_price < 0")
    .withColumn("rejection_reason", F.lit("Invalid product ID or price"))
)

# Write valid products to Silver layer
(valid_products.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.products"))
# Write invalid products to quarantine table
(invalid_products.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.products_quarantine"))

display(valid_products)
display(invalid_products)

product_id,product_name,category,unit_price,_ingested_at
101,Laptop,Electronics,1500.00,2026-09-08T17:52:29.732Z
102,Mouse,null,25.00,2026-09-08T17:52:29.732Z
103,Keyboard,Electronics,49.99,2026-09-08T17:52:29.732Z


product_id,product_name,category,unit_price,_ingested_at,rejection_reason
104,Desk,Office Furniture,null,2026-09-08T17:52:29.732Z,Invalid product ID or price


## 6. Silver Layer: Validate Orders

Rules:

- Convert IDs and dates to proper types.
- Standardize order status.
- Accept only known customers.
- Reject future dates and malformed records.

In [0]:
# Load raw orders from Bronze layer
orders_raw = spark.table(f"{BRONZE}.orders")
# Get list of valid customer IDs from Silver customers for referential integrity check
valid_customer_ids = spark.table(f"{SILVER}.customers").select("customer_id")

# Clean and standardize order fields
orders_staged = (
    orders_raw
    .select(
        F.col("order_id").cast("int").alias("order_id"),  # Cast ID to integer
        F.col("customer_id").cast("int").alias("customer_id"),  # Cast customer ID to integer
        F.to_date("order_date").alias("order_date"),  # Parse date
        # Standardize status values to title case
        F.when(F.lower(F.trim("status")).isin("complete", "completed"), "Completed")
         .when(F.lower(F.trim("status")) == "pending", "Pending")
         .otherwise(F.initcap(F.trim("status"))).alias("status"),
        "_ingested_at"  # Preserve metadata
    )
)

# Join with valid customers to flag orders with known customers
orders_checked = orders_staged.join(
    valid_customer_ids.withColumn("customer_exists", F.lit(True)),
    "customer_id", "left"
)

# Filter for valid orders: non-null IDs/dates, non-future dates, and known customers
valid_orders = orders_checked.filter(
    (F.col("order_id").isNotNull()) &
    (F.col("order_date").isNotNull()) &
    (F.col("order_date") <= F.current_date()) &
    (F.col("customer_exists") == True)
).drop("customer_exists")

# Quarantine invalid orders with rejection reasons for review
invalid_orders = (
    orders_checked
    .filter(
        F.col("order_id").isNull() |
        F.col("order_date").isNull() |
        (F.col("order_date") > F.current_date()) |
        F.col("customer_exists").isNull()
    )
    .withColumn(
        "rejection_reason",
        F.when(F.col("customer_exists").isNull(), "Unknown customer")
         .when(F.col("order_date") > F.current_date(), "Future order date")
         .otherwise("Invalid order record")
    )
    .drop("customer_exists")
)

# Write valid orders to Silver layer
(valid_orders.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.orders"))
# Write invalid orders to quarantine table
(invalid_orders.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.orders_quarantine"))

# Display results for review
display(valid_orders)
display(invalid_orders)

customer_id,order_id,order_date,status,_ingested_at
1,1001,2026-01-15,Completed,2026-09-08T17:52:32.368Z
2,1002,2026-01-16,Completed,2026-09-08T17:52:32.368Z


customer_id,order_id,order_date,status,_ingested_at,rejection_reason
99,1003,2026-01-17,Completed,2026-09-08T17:52:32.368Z,Unknown customer
3,1004,2099-01-01,Pending,2026-09-08T17:52:32.368Z,Future order date


## 7. Silver Layer: Validate and Enrich Order Items

Rules:

- Remove exact duplicates.
- Require a positive quantity.
- Require matching Silver orders and products.
- Add the product price and calculate `line_total`.

In [0]:
# Load raw order items from Bronze and cast types
items_raw = (
    spark.table(f"{BRONZE}.order_items")
    .select(
        F.col("order_id").cast("int").alias("order_id"),
        F.col("product_id").cast("int").alias("product_id"),
        F.col("quantity").cast("int").alias("quantity"),
        "_ingested_at"
    )
    .dropDuplicates(["order_id", "product_id", "quantity"])  # Remove exact duplicates
)

# Get valid order IDs and product details for referential integrity checks
orders_keys = spark.table(f"{SILVER}.orders").select("order_id")
product_lookup = spark.table(f"{SILVER}.products").select(
    "product_id", "product_name", "category", "unit_price"
)

# Join with orders and products to validate and enrich
items_checked = (
    items_raw
    .join(orders_keys.withColumn("order_exists", F.lit(True)), "order_id", "left")
    .join(product_lookup, "product_id", "left")  # Adds unit_price for line total calculation
)

# Filter for valid items: positive quantity, known order, and known product
valid_items = (
    items_checked
    .filter(
        (F.col("quantity") > 0) &
        (F.col("order_exists") == True) &
        F.col("unit_price").isNotNull()
    )
    .withColumn("line_total", F.col("quantity") * F.col("unit_price"))  # Calculate revenue
    .drop("order_exists")
)

# Quarantine invalid items with rejection reasons
invalid_items = (
    items_checked
    .filter(
        (F.col("quantity") <= 0) |
        F.col("quantity").isNull() |
        F.col("order_exists").isNull() |  # Order not found in Silver
        F.col("unit_price").isNull()  # Product not found in Silver
    )
    .withColumn(
        "rejection_reason",
        F.when(F.col("quantity") <= 0, "Quantity must be positive")
         .when(F.col("order_exists").isNull(), "Unknown order")
         .when(F.col("unit_price").isNull(), "Unknown product")
         .otherwise("Invalid order item")
    )
    .drop("order_exists")
)

# Write valid items to Silver layer
(valid_items.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.order_items"))
# Write invalid items to quarantine for review
(invalid_items.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.order_items_quarantine"))

display(valid_items)
display(invalid_items)

product_id,order_id,quantity,_ingested_at,product_name,category,unit_price,line_total
101,1001,1,2026-09-08T17:52:35.012Z,Laptop,Electronics,1500.00,1500.00
102,1001,2,2026-09-08T17:52:35.012Z,Mouse,null,25.00,50.00
103,1002,3,2026-09-08T17:52:35.012Z,Keyboard,Electronics,49.99,149.97


product_id,order_id,quantity,_ingested_at,product_name,category,unit_price,rejection_reason
999,1002,1,2026-09-08T17:52:35.012Z,null,null,null,Unknown product
103,1003,-1,2026-09-08T17:52:35.012Z,Keyboard,Electronics,49.99,Quantity must be positive


## 8. Validate the Silver Layer

The checks below fail fast if Silver violates core quality expectations.

In [0]:
# Load Silver tables
customers = spark.table(f"{SILVER}.customers")
products = spark.table(f"{SILVER}.products")
orders = spark.table(f"{SILVER}.orders")
items = spark.table(f"{SILVER}.order_items")

# No duplicate customer IDs
assert customers.groupBy("customer_id").count().filter("count > 1").count() == 0
# All prices are valid (non-null and non-negative)
assert products.filter("unit_price IS NULL OR unit_price < 0").count() == 0
# No future order dates
assert orders.filter(F.col("order_date") > F.current_date()).count() == 0
# All quantities and totals are positive
assert items.filter("quantity <= 0 OR line_total < 0").count() == 0
# All orders reference valid customers
assert orders.join(customers, "customer_id", "left_anti").count() == 0
# All items reference valid orders
assert items.join(orders, "order_id", "left_anti").count() == 0
# All items reference valid products
assert items.join(products, "product_id", "left_anti").count() == 0

print("All Silver quality checks passed.")

All Silver quality checks passed.


## 9. Build a Reusable Silver Sales View

Join trusted Silver tables at transaction-line grain: one row per order and product.

In [0]:
# Join order items with orders and customers to create a complete sales transaction view
sales_detail = (
    spark.table(f"{SILVER}.order_items").alias("i")
    .join(spark.table(f"{SILVER}.orders").alias("o"), "order_id")  # Add order date and status
    .join(spark.table(f"{SILVER}.customers").alias("c"), "customer_id")  # Add customer details
    .select(
        "order_id", "customer_id", "customer_name", "country_code",
        "order_date", "status", "product_id", "product_name", "category",
        "quantity", "unit_price", "line_total"
    )
)

# Write the denormalized sales view to Silver as a reusable table
(sales_detail.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.sales_detail"))
display(sales_detail)

order_id,customer_id,customer_name,country_code,order_date,status,product_id,product_name,category,quantity,unit_price,line_total
1001,1,Ali Hassan,QAT,2026-01-15,Completed,101,Laptop,Electronics,1,1500.00,1500.00
1001,1,Ali Hassan,QAT,2026-01-15,Completed,102,Mouse,null,2,25.00,50.00
1002,2,Sara Ahmed,QAT,2026-01-16,Completed,103,Keyboard,Electronics,3,49.99,149.97


## 10. Gold Layer: Business-Ready Metrics

Create three simple Gold tables:

1. Daily sales summary
2. Product performance
3. Customer summary

In [0]:
# Load the clean, denormalized sales transaction table from Silver
sales = spark.table(f"{SILVER}.sales_detail")

# Aggregate daily sales: orders, units, and revenue by date
gold_daily_sales = (
    sales.groupBy("order_date")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("line_total"), 2).alias("total_sales")
    )
)

# Aggregate product performance: units, orders, and revenue by product
gold_product_performance = (
    sales.groupBy("product_id", "product_name", "category")
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.countDistinct("order_id").alias("orders_count"),
        F.round(F.sum("line_total"), 2).alias("revenue")
    )
)

# Aggregate customer summary: orders, units, and spending by customer
gold_customer_summary = (
    sales.groupBy("customer_id", "customer_name", "country_code")
    .agg(
        F.countDistinct("order_id").alias("orders_count"),
        F.sum("quantity").alias("units_purchased"),
        F.round(F.sum("line_total"), 2).alias("total_spent")
    )
)

# Write each Gold table to Unity Catalog (overwrite mode for idempotency)
for name, df in {
    "daily_sales": gold_daily_sales,
    "product_performance": gold_product_performance,
    "customer_summary": gold_customer_summary,
}.items():
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{GOLD}.{name}"))

print("Gold tables created.")

Gold tables created.


In [0]:
# Display daily sales metrics sorted chronologically
display(spark.table(f"{GOLD}.daily_sales").orderBy("order_date"))

# Display product performance sorted by highest revenue first
display(spark.table(f"{GOLD}.product_performance").orderBy(F.desc("revenue")))

# Display customer summary sorted by highest spending first
display(spark.table(f"{GOLD}.customer_summary").orderBy(F.desc("total_spent")))

order_date,total_orders,units_sold,total_sales
2026-01-15,1,3,1550.00
2026-01-16,1,3,149.97


product_id,product_name,category,units_sold,orders_count,revenue
101,Laptop,Electronics,1,1,1500.00
103,Keyboard,Electronics,3,1,149.97
102,Mouse,null,2,1,50.00


customer_id,customer_name,country_code,orders_count,units_purchased,total_spent
1,Ali Hassan,QAT,1,3,1550.00
2,Sara Ahmed,QAT,1,3,149.97


## 11. Final Architecture and Tables

```text
External CSV files (Volume)
   ↓
Bronze: raw Delta tables
   ↓
Silver: clean + standardize + validate + enrich
   ↓
Gold: aggregate metrics for reporting
```

Review all created tables:

In [0]:
%sql
SHOW TABLES IN sales_demo.bronze;

database,tableName,isTemporary
bronze,customers,false
bronze,order_items,false
bronze,orders,false
bronze,products,false


In [0]:
%sql
SHOW TABLES IN sales_demo.silver;

database,tableName,isTemporary
silver,customers,false
silver,order_items,false
silver,order_items_quarantine,false
silver,orders,false
silver,orders_quarantine,false
silver,products,false
silver,products_quarantine,false
silver,sales_detail,false


In [0]:
%sql
SHOW TABLES IN sales_demo.gold;

database,tableName,isTemporary
gold,customer_summary,false
gold,daily_sales,false
gold,product_performance,false


## 12. Import and Run on Databricks

1. Save this notebook as `retail_sales_medallion_databricks.ipynb`.
2. Sign in to the Databricks workspace.
3. Open **Workspace** and choose your user folder or a training folder.
4. Select **Import**, then upload the `.ipynb` file.
5. Open the imported notebook and run the setup cells to create the catalog, schemas, and the `raw_files` volume.
6. Upload the four CSV files from the `data/` folder to the `sales_demo.bronze.raw_files` volume via **Catalog Explorer**.
7. Select **Run all**.
8. Open **Catalog Explorer** and inspect:
   - `sales_demo.bronze`
   - `sales_demo.silver`
   - `sales_demo.gold`
9. Review the Silver quarantine tables to understand rejected records.
10. Re-run the notebook to demonstrate idempotency. The tutorial uses overwrite mode, so repeated runs produce the same logical result.

### Optional Reset

Run the following only when you want to remove the complete tutorial environment:

```sql
DROP CATALOG IF EXISTS sales_demo CASCADE;
```